In [ ]:
# Import SparkSession and initialize Spark application
from pyspark.sql import SparkSession
spark=(
    SparkSession
    .builder
    .appName("Skewness & salting technique")
    .master("local[*]")
    .config("spark.executor.cores",4)
    .config("spark.executor.memory","512M")
    .getOrCreate()
)

In [2]:
spark


In [3]:
# Disable AQE and Broadcast join

spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [5]:
# Read the employee_rec dataset
_schema="first_name string,last_name string,job_title string,dob string,email string,phone string,salary string,department_id string"
emp=spark.read.format("csv").schema(_schema).option("header",True).load("employee_rec.csv")

In [7]:
# read the department data
_schema="department_id string,department_name string,description string,city string,state string,country string"
dept=spark.read.format("csv").schema(_schema).option("header",True).load("department_data.csv")

In [ ]:
# joining emp&dept with department_id on left_outer join
df_joined=emp.join(dept,on=emp.department_id == dept.department_id,how="left_outer")

In [ ]:
# Save dataframe 
df_joined.write.format("noop").mode("overwrite").save()

In [14]:
from pyspark.sql.functions import spark_partition_id , count,lit
part_df=df_joined.withColumn("partition_num",spark_partition_id()).groupBy("partition_num").agg(count(lit(1)).alias("count"))

In [15]:
part_df.show()

+-------------+------+
|partition_num| count|
+-------------+------+
|          155| 99780|
|           26|100417|
|            3| 99805|
|          139|100014|
|          189|100155|
|           49|100210|
|          166|100214|
|           75| 99706|
|          144| 99451|
|           18|100248|
+-------------+------+



In [22]:
# verify department data based on dept_id
from pyspark.sql.functions import desc,count,lit
emp.groupBy("department_id").agg(count(lit(1))).show()

+-------------+--------+
|department_id|count(1)|
+-------------+--------+
|            7|   99805|
|            3|  100248|
|            8|  100417|
|            5|  100210|
|            6|   99706|
|            9|  100014|
|            1|   99451|
|           10|   99780|
|            4|  100214|
|            2|  100155|
+-------------+--------+



In [ ]:
# set shuffle partition to 16
spark.conf.set("spark.sql.shufflepartitions",16)

In [ ]:
# creating the random numbers
import random
from pyspark.sql.functions import udf
@udf
def salt_udf():
    return random.randint(0,16)
    

In [38]:
salt_df=spark.range(0,16)
salt_df.show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
| 10|
| 11|
| 12|
| 13|
| 14|
| 15|
+---+



In [ ]:
# Check skew in employee data by department_id
from pyspark.sql.functions import concat,lit
salted_emp=emp.withColumn("salted_dept_id",concat("department_id",lit("-"),salt_udf()))

In [43]:
salted_dept=dept.join(salt_df,how="cross").withColumn("salted_dept_id",concat("department_id",lit("-"),"id"))

In [45]:
salted_dept.where("department_id=1").show()

+-------------+---------------+--------------------+------------+-----+-------------------+---+--------------+
|department_id|department_name|         description|        city|state|            country| id|salted_dept_id|
+-------------+---------------+--------------------+------------+-----+-------------------+---+--------------+
|            1|    Bryan-James|Optimized disinte...|Melissaburgh|   FM|Trinidad and Tobago|  0|           1-0|
|            1|    Bryan-James|Optimized disinte...|Melissaburgh|   FM|Trinidad and Tobago|  1|           1-1|
|            1|    Bryan-James|Optimized disinte...|Melissaburgh|   FM|Trinidad and Tobago|  2|           1-2|
|            1|    Bryan-James|Optimized disinte...|Melissaburgh|   FM|Trinidad and Tobago|  3|           1-3|
|            1|    Bryan-James|Optimized disinte...|Melissaburgh|   FM|Trinidad and Tobago|  4|           1-4|
|            1|    Bryan-James|Optimized disinte...|Melissaburgh|   FM|Trinidad and Tobago|  5|           1-5|
|

In [47]:
salted_join_df=salted_emp.join(salted_dept,on=salted_emp.salted_dept_id==salted_dept.salted_dept_id,how="left_outer")

In [48]:
salted_join_df.write.format("noop").mode("overwrite").save()

In [50]:
from pyspark.sql.functions import spark_partition_id , count,lit
part_df=salted_join_df.withColumn("partition_num",spark_partition_id()).groupBy("partition_num").agg(count(lit(1)).alias("count"))

In [59]:
part_df.show(1000)

+-------------+-----+
|partition_num|count|
+-------------+-----+
|           85| 5908|
|          137| 5984|
|          133| 5914|
|           78|11788|
|           34| 5971|
|          193| 5963|
|          115| 5883|
|          126| 5850|
|           81| 5861|
|           76|11618|
|           26| 5820|
|           27| 5722|
|           44| 5906|
|           12| 5806|
|           22| 5902|
|          128|11646|
|           93|11897|
|          157|11721|
|          190|23745|
|          111| 5855|
|           47| 5939|
|          185|11753|
|          146| 5691|
|            1| 5842|
|           52|11563|
|          182|11607|
|           13| 5732|
|            6| 5848|
|           16| 5845|
|           86|11792|
|          168| 5836|
|           20| 5844|
|           94|11825|
|           57| 5889|
|           54| 5918|
|           96|17831|
|            5|11896|
|          163| 5941|
|           19|11830|
|           64|17661|
|          117|11796|
|          154| 5906|
|         